# Projeto Final Análise de Texto de Fontes Desestruturadas e Web.

Integrantes do Grupo:


*   Brian Lee Chan
*  Felipe Lima
*   Henrique Rocha Bomfim
*   Yasmin Tessari Primo




# **Projeto Garimpa Ae**


# Introdução

Garimpa Aí é um projeto que visa desenvolver uma ferramenta flexível de coleta e análise de preços em marketplaces.

Queremos criar um mecanismo capaz de acessar automaticamente páginas de busca em grandes marketplaces (por exemplo, Amazon, Mercado Livre, Americanas), navegar por múltiplas páginas de resultados e compilar todas as ofertas de um determinado produto em um único banco de dados.

Visamos utilizar a biblioteca Selenium em modo headless para lidar com páginas que carregam conteúdo via JavaScript, garantindo que nenhuma oferta seja perdida quando, por exemplo, o preço ou a lista de vendedores só aparece após a página ser totalmente renderizada.

## Problema

Em um cenário econômico cada vez mais desafiador, seja em função da inflação ou da flutuação cambial, consumidores buscam constantemente a melhor oferta antes de fechar uma compra. Desde estudantes montando seus primeiros setups de computador até famílias planejando a compra de eletrodomésticos, a necessidade de “garimpar” — isto é, encontrar o produto certo pelo menor preço possível — transformou-se em um hábito diário.

Ao mesmo tempo, a internet se tornou uma fonte inesgotável de opções: marketplaces repletos de milhares de vendedores, cada um ofertando modelos ligeiramente diferentes, promoções relâmpago, descontos por tempo limitado e variantes regionais. Nesse ambiente dinâmico, tomar a decisão de compra “no olho” é ineficiente e, muitas vezes, frustrante, pois é fácil perder a melhor oportunidade de preço, perdendo aquele descontão.

É exatamente nesse ponto que a extração automática de dados ganha relevância. Se pudermos desenvolver alguma ferramenta que captura instantaneamente todas as ofertas de um produto, com suas especificações e valores, é possível comparar centenas de vendedores em segundos, em vez de um bom tempo de navegação manual. O ato de “garimpar preços” — reunir relatividades, flutuações e avaliações de forma estruturada — empodera o consumidor a tomar decisões conscientes, economizar tempo e, sobretudo, economizar dinheiro.
Por isso, ao fazer isso, pegaremos de uma maneira mais rápida e eficiente os melhores preços de dois dos maiores marketplaces do Brasil, mostrando também a diferença deles para o mesmo produto. Um sendo referência Internacional de negócios, e o outro, passando por tempos difíceis para manter relevancia no mercado.

Para efetivamente fazer nosso trabalho, precisamos fazer um Scrapper. Um scrapper nada mais é do que um script que vai essencialmente extrair os dados de websites automaticamente. Iremos extrair de dois sites específicos. Amazon e Americanas

Referências:

(Além das notas de Aula e Atividades)

1.   ChatGPT
2.   https://stackoverflow.com/questions/79646981/opening-site-only-possible-in-non-headless-mode
3.   https://stackoverflow.com/questions/79640391/persistent-nosuchelementexception-with-empty-csv-output-in-selenium-web-scraping
4. https://stackoverflow.com/questions/76795060/i-cant-extract-info-using-class-selector-on-selenium-webscrapper
5. https://selenium-python.readthedocs.io/navigating.html#moving-between-windows-and-frames
6. https://selenium-python.readthedocs.io/locating-elements.html#locating-hyperlinks-by-link-text



# Extração dos dados

Primeiramente, é recomendado utilizar o google collab no google chrome para evitar falta de compatibilidade e incoêrencias com o Selenium.

Instalações das bibliotecas utilizadas

In [1]:
!python -m pip install -q selenium webdriver_manager beautifulsoup4 pandas requests
!sudo apt-get update -qq
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!sudo dpkg -i google-chrome-stable_current_amd64.deb
!sudo apt-get -f install -qq -y
!google-chrome-stable --version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 5.5 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package google-chrome-stable.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack google-chrome-stable_current_amd64.deb ...
Unpacking google-chrome-stable (145.0.7632.159-1) ...
dpkg: dependency problems prevent configuration of google-chrome-stable:
 google-chrome-stable depends on libatk-bridge2.0-0 (>= 2.5.3); however:
  Package libatk-bridge2.0-0 is not installed.
 google-chrome-stable depends on libatk1.0-0 (>= 2.11.90); however:
  Package libatk1.0-0 is not installed.
 google-chrome-stabl

In [2]:
!python -m pip show webdriver_manager

Name: webdriver-manager
Version: 4.0.2
Summary: Library provides the way to automatically manage drivers for different browsers
Home-page: https://github.com/SergeyPirogov/webdriver_manager
Author: Sergey Pirogov
Author-email: automationremarks@gmail.com
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: packaging, python-dotenv, requests
Required-by: 


Importações de todas as bibliotecas utilizadas ao longo do projeto, além da configuração do Selenium, permitindo-o trabalhar sem a necessidade de abrir abas do usuario.

In [3]:
#Todas as importacoes que iremos precisar
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
# Requests + BeautifulSoup: para parsing de páginas caso necessário
import requests
from bs4 import BeautifulSoup
# Pandas: para manipulação de tabelas e exportação em CSV
import pandas as pd
# Time e Random: para controlar delays “humanos” entre requisições
import time
import random
# OS: para operações de sistema (listar arquivos, salvar CSV, etc.)
import os
# RE: expressões regulares, usadas na limpeza de preços e extração de números
import re
import math

In [4]:
# 1) Configuração do Chrome em modo headless (essencial no Colab)
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")

# Adiciona User-Agent realista (imitando navegador real)
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0 Safari/537.36"
)

options.binary_location = "/usr/bin/google-chrome-stable"
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

print("✅ Selenium + Google Chrome (headless) inicializado com sucesso!")

✅ Selenium + Google Chrome (headless) inicializado com sucesso!


Aqui a primeira função, precisamos de alguma maneira para abrir a amazon e a americanas, e navegar por elas sem que sejamos acusados de BOT's pelo sistema, para fazer isso, além de acessar o url temos que simular um comprtamento humano, tal qual é feito utilizando um "time_sleep" de maneira aleatoria, assim simulando o tempo que um ser humano levaria para passar de pagina ou entender os conteudos que estão sendo mostrados na tela.

In [ ]:
def abre_amazon_para_busca(produto, page=1):
    """
    Abre a URL de busca da Amazon usando Selenium e retorna o HTML completo (com JS já executado).
    Parâmetros:
      • produto (str): termo de busca, ex. "Samsung Galaxy A53"
      • page (int)   : número da página de resultados (1, 2, 3, …)
    Retorno:
      • html (str): HTML completo da página, pronto para ser parseado com BeautifulSoup.
    """
    termo = produto.replace(" ", "+")
    url_busca = f"https://www.amazon.com.br/s?k={termo}&page={page}"
    print("URL de busca:", url_busca)
    driver.get(url_busca)

    # Espera “humana” entre 4 e 10 segundos para o JS carregar
    time.sleep(random.uniform(4, 10))
    #FAZENDO ISSO POR QUE A AMAZON ESTAVA DANDO O ERRO 503, que estava essencialmente bloqueando, como um mecanismo "anti scraping"
    #aqui retornando o html
    return driver.page_source

def abre_americanas_para_busca(produto, page=0):
    """
    Abre a URL de busca da Americanas usando Selenium e retorna o HTML completo (com JS já executado).
    Parâmetros:
      • produto (str): termo de busca, ex. "iPhone 14"
      • page (int)   : número da página de resultados (0, 1, 2, …)
    Retorno:
      • html (str): HTML completo da página, pronto para ser parseado com BeautifulSoup.
    """
    termo = produto.replace(" ", "+")
    url_busca = f"https://www.americanas.com.br/s?q={termo}&sort=score_desc&page={page}"
    print("URL de busca:", url_busca)
    driver.get(url_busca)

    # Espera “humana” entre 6 e 8 segundos para o JS carregar
    time.sleep(random.uniform(6, 8))

    # Retorna o HTML da página
    return driver.page_source

Para continuar, é necessario (mais) funções, dessa vez precisamos de algo que  torna possível padronizar e filtrar o texto extraído dos sites de forma automática antes de qualquer comparação ou análise.

Primeiramente, o remove_acentos_regex elimina acentos para uniformizar palavras em português e evitar o irreconhecimento de termos. Em seguida, o clean_price converte qualquer representação de preço (como “R$ 2.999,00”) em um valor numérico, garantindo que possamos ordená-los e calculá-los corretamente.

Já a função clean_title permite que nós conssigamos simplificar títulos removendo "quebras de linha" e espaços indevidos, para que comparações não sejam afetadas por formatações inconsistentes. Com textos normalizados conseguimos evitar erros e especialemnte implementar outra função!. A função calculate_similarity mede a proporção de palavras em comum entre o termo buscado e o título do produto, permitindo descartar itens com baixa correspondência. Antes de aceitar um resultado, o contains_blacklisted_keyword verifica se o título contém alguma expressão indesejada (como “capa” ou “carregador”), e o validate_product_relevance combina essa checagem de blacklist com o critério de similaridade mínima para só manter ofertas realmente relevantes ao que o usuário procura.

Com elas, conseguimos de maneira mais apropriada, guiar cada vez mais o scrapper de maneira que consigamos evitar os problemas nas buscas dos preços e produtos. Mas ainda assim, precisamos de mais algumas coisas...

((Utilizamos RegEx para garantir que independente da forma que o usuário digite, o filtro e o produto serão bem interpretados pelo nosso programa, permitindo que câmera e camera sejam avaliados da mesma maneira, por exemplo.))

In [6]:
def remove_acentos_regex(texto):
    # Substituindo letras acentuadas por letras sem acento
    texto = re.sub(r'[áàâãä]', 'a', texto, flags=re.IGNORECASE)
    texto = re.sub(r'[éèêë]', 'e', texto, flags=re.IGNORECASE)
    texto = re.sub(r'[íìîï]', 'i', texto, flags=re.IGNORECASE)
    texto = re.sub(r'[óòôõö]', 'o', texto, flags=re.IGNORECASE)
    texto = re.sub(r'[úùûü]', 'u', texto, flags=re.IGNORECASE)
    texto = re.sub(r'[ç]', 'c', texto, flags=re.IGNORECASE)
    return texto

def clean_price(price_text):
    """
    Converte texto de preço (ex.: "R$ 2.999,00") em float (2999.0).
    """
    if not price_text:
        return None
    #limpando completamente com regex
    text = re.sub(r"[^\d,\.]", "", price_text.replace("R$", "").strip())
    # padronizando o valores
    if "," in text and "." in text:
        text = text.replace(".", "").replace(",", ".")
    elif "," in text:
        text = text.replace(",", ".")
    try:
        return float(text)
    except:
        return None

def clean_title(title):
    """
    Remove quebras de linha e espaços duplicados do título.
    """
    return re.sub(r"\s+", " ", title.strip()) if title else ""

def calculate_similarity(produto_buscado, titulo_encontrado):
    """
    Calcula (|palavras em comum| / |palavras do termo buscado|).
    Retorna valor entre 0.0 e 1.0.
    """
    produto_words = set(remove_acentos_regex(produto_buscado).lower().split())
    titulo_words  = set(remove_acentos_regex(titulo_encontrado).lower().split())
    #print(titulo_words)
    if not produto_words:
        return 0
    intersec = produto_words.intersection(titulo_words)
    return len(intersec) / len(produto_words)

def contains_blacklisted_keyword(titulo, blacklist):
    """
    Retorna True se 'titulo' (em minúsculas) contiver qualquer palavra de 'blacklist' (lista de strings).
    """
    if not titulo:
        return False
    tl = remove_acentos_regex(titulo).lower()
    for kw in blacklist:
        if remove_acentos_regex(kw).lower() in tl:
            return True
    return False

def validate_product_relevance(produto_buscado, titulo_produto, min_similarity, blacklist):
    """
    1) Descarta se o título contiver alguma palavra da 'blacklist'.
    2) Caso contrário, calcula similaridade; retorna True se >= min_similarity.
    """
    if contains_blacklisted_keyword(titulo_produto, blacklist):
        return False
    sim = calculate_similarity(produto_buscado, titulo_produto)
    return sim >= min_similarity


Para continuar o nosso processo, precisamos criar outras duas funções que permitam a automatização e a captura de ofertas de produtos em sites de e-commerce (Amazon e Americanas) usando o Selenium para garantir que todo o conteúdo dinâmico seja carregado antes de extrair informações.

Primeiro, cada função recebe como entrada o termo de busca (no caso o produto que o usuario deseja buscar), a quantidade de páginas a percorrer (caso seja um produto mais específico, talvez aumentar esse numero seja uma boa ideia), um limiar de similaridade (mesma ideia) e uma lista de palavras extras a descartar. A partir daí, para cada página, montamos a URL adequada, carregando a pagina no navegador headless como configuramos no começo e aguardam alguns segundos de forma aleatória, simulando comportamento humano e evitando bloqueios (que foram uma das maiores dificuldades no começo do scrapping).

Após renderizar tudo, utilizamos o BeautifulSoup para localizar todos os “containers” de produtos (blocos que representam cada oferta) e, em cada um, extraem título, preço e link (além de avaliação e número de reviews na Amazon, quando disponíveis). Antes de aceitar um item, aplicamos os dois critérios de filtragem: DESCARTANDO títulos que contenham termos indesejados da blacklist combinada (lista base mais as palavras extras que o usuário passou) e verificam se a similaridade textual entre o termo buscado e o título do produto é suficientemente alta. Ao final de cada página, há outra pausa aleatória para manter o scraping discreto.

 O resultado final de cada função é uma lista de dicionários com os campos principais – título, preço, link, similaridade e página –, isso no qual é o começo do nosso produto final, que eventualemte será ordenada, exibida ao usuário e exportada em CSV para facilitar a comparação de preços

In [7]:
def scrape_amazon_selenium(
    produto,
    max_pages=3,
    min_similarity=0.3,
    extra_blacklist=None
):
    """
    Faz scraping na Amazon usando Selenium + BeautifulSoup + filtros dinâmicos.

    Parâmetros:
      produto (str): termo de busca, ex.: "Samsung Galaxy A53"
      max_pages (int): número máximo de páginas a percorrer
      min_similarity (float): similaridade mínima exigida (0.0–1.0)
      extra_blacklist (list[str] | None): palavras extras para descartar

    Retorna:
      List[dict] onde cada dict contém:
        {
          "titulo": str,
          "preco": float,
          "avaliacao": float | None,
          "num_avaliacoes": int | None,
          "link": str | None,
          "similaridade": float,
          "pagina": int
        }
    """
    resultados = []

    print(f"🔍 Buscando (Selenium) por: '{produto}'  |  Páginas: {max_pages}  |  Sim. mínima: {min_similarity:.1%}")
    print("-" * 60)

    # 6.1) montando a blacklist combinada (BASE + extras)
    combined_blacklist = extra_blacklist

    for page in range(1, max_pages + 1):
        # 6.2)abrindo e pegando o html igual em aula
        print(f"· Página {page}...", end=" ")
        html = abre_amazon_para_busca(produto, page)
        soup = BeautifulSoup(html, "html.parser")

        # 6.3) realmente nao sei oq essa parte faz direito.
        selectors = [
            'div[data-component-type="s-search-result"]',
            '.s-result-item',
            '[data-asin]'
        ]
        product_containers = []
        for sel in selectors:
            product_containers = soup.select(sel)
            if product_containers:
                break

        if not product_containers:
            print("sem produtos.")
            time.sleep(random.uniform(2, 5))
            continue

        print(f"{len(product_containers)} encontrados.")
        for container in product_containers:
            try:
                # (a) extraindo título
                title = None
                title_selectors = [
                    'h2 a span',
                    'h2 span',
                    '.a-size-base-plus',
                    '.a-size-medium',
                    '.s-size-mini'
                ]
                for sel in title_selectors:
                    elem = container.select_one(sel)
                    if elem and elem.get_text().strip():
                        title = clean_title(elem.get_text())
                        break
                if not title or len(title) < 5:
                    continue

                #filtrando pela blacklist e similaridade
                if not validate_product_relevance(produto, title, min_similarity, combined_blacklist):
                    continue

                # (c) extraindo o preco
                price = None
                price_selectors = [
                    '.a-price-whole',
                    '.a-price .a-offscreen',
                    '.a-price-range .a-offscreen',
                    'span[data-a-color="price"]',
                    '.a-price-symbol + .a-price-whole'
                ]
                for sel in price_selectors:
                    pelem = container.select_one(sel)
                    if pelem:
                        price = clean_price(pelem.get_text())
                        if price and price > 0:
                            break

                # (d) Extrair link via <a href>
                link = None
                link_elem = container.select_one("h2 a")
                if link_elem:
                    href = link_elem.get("href", "")
                    if href.startswith("/"):
                        link = "https://www.amazon.com.br" + href

                # (e) Se não obteve link, monta via ASIN (ele nao tava retornando o link algumas vezes entao essa solucao funcionou)
                if not link:
                    asin = container.get("data-asin", "").strip()
                    if asin:
                        link = f"https://www.amazon.com.br/dp/{asin}"

                # (f) pegando as estrelas para ficar mais vizual ja que nao ta dando certo retornar aquela tabelona q comentamos
                rating = None
                rating_elem = container.select_one(".a-icon-alt")
                if rating_elem:
                    m = re.search(r"(\d+[.,]?\d*)", rating_elem.get_text())
                    if m:
                        try:
                            rating = float(m.group(1).replace(",", "."))
                        except:
                            rating = None

                # (g) extraindo as reviews tambem
                num_reviews = None
                review_selectors = [
                    'a[href*="#customerReviews"] span',
                    ".a-size-base",
                    'span[aria-label*="estrelas"]'
                ]
                for sel in review_selectors:
                    rev = container.select_one(sel)
                    if rev:
                        m2 = re.search(r"(\d+)", rev.get_text().replace(".", "").replace(",", ""))
                        if m2:
                            try:
                                num_reviews = int(m2.group(1))
                                break
                            except:
                                pass

                # (h) aqui adicionando nos resultados
                if title and price and price > 0:
                    resultados.append({
                        "titulo": title[:100] + ("..." if len(title) > 100 else ""),
                        "preco": price,
                        "avaliacao": rating,
                        "num_avaliacoes": num_reviews,
                        "link": link,
                        "similaridade": calculate_similarity(produto, title),
                        "pagina": page
                    })

            except Exception:
                # Se falhar ao parsear este container, pula para o próximo
                continue

        # 6.5) pausa “humana” antes da próxima página, para que o "robo" não seja bloqueado pela página WEB
        time.sleep(random.uniform(2, 5))

    return resultados

def scrape_americanas_selenium(
    produto,
    max_pages=3,
    min_similarity=0.3,
    extra_blacklist=None
):
    """
    Faz scraping na Americanas usando Selenium + BeautifulSoup + filtros dinâmicos.

    Parâmetros:
      produto (str): termo de busca, ex.: "Samsung Galaxy A53"
      max_pages (int): número máximo de páginas a percorrer
      min_similarity (float): similaridade mínima exigida (0.0–1.0)
      extra_blacklist (list[str] | None): palavras extras para descartar

    Retorna:
      List[dict] onde cada dict contém:
        {
          "titulo": str,
          "preco": float,
          "link": str | None,
          "similaridade": float,
          "pagina": int
        }
    """
    resultados = []

    print(f"🔍 Buscando (Selenium) por: '{produto}'  |  Páginas: {max_pages}  |  Sim. mínima: {min_similarity:.1%}")
    print("-" * 60)

    combined_blacklist = extra_blacklist

    for page in range(max_pages):
        print(f"· Página {page}...", end=" ")
        html = abre_americanas_para_busca(produto, page)
        soup = BeautifulSoup(html, "html.parser")

        # Ajustado: seletores corretos para os containers de produtos na Americanas
        product_containers = soup.find_all("div", class_=lambda c: c and "ProductCard_productInfo" in c)

        if not product_containers:
            print("sem produtos.")
            time.sleep(random.uniform(2, 5))
            continue

        print(f"{len(product_containers)} encontrados.")
        for container in product_containers:
            try:
                # (a) Título do produto
                titulo_elem = container.select_one("h3")
                if not titulo_elem:
                    continue
                title = clean_title(titulo_elem.get_text())
                if not title or len(title) < 5:
                    continue

                # Filtrando pela blacklist e similaridade
                if not validate_product_relevance(produto, title, min_similarity, combined_blacklist):
                    continue

                # (b) Preço
                price = None
                price_elem = container.select_one('span[class*="ProductCard_discountPrice"]')
                if price_elem:
                    price = clean_price(price_elem.get_text())
                if not price or price <= 0:
                    continue

                # (c) Link do produto
                link = None
                # O link geralmente está no ancestor 'a' do container de info
                link_elem = container.find_parent("a", href=True)
                if link_elem:
                    href = link_elem.get("href", "")
                    if href.startswith("/"):
                        link = "https://www.americanas.com.br" + href
                    else:
                        link = href

                # (d) Monta resultado final
                resultados.append({
                    "titulo": title[:100] + ("..." if len(title) > 100 else ""),
                    "preco": price,
                    "avaliacao": None,          # Americanas não mostra avaliação visível
                    "num_avaliacoes": None,     # Americanas não mostra reviews visíveis
                    "link": link,
                    "similaridade": calculate_similarity(produto, title),
                    "pagina": page
                })

            except Exception:
                continue

        time.sleep(random.uniform(2, 5))

    return resultados


Chegando cada vez mais perto de um mecanismo funcional, precisamos de algo que consiga analisar os nossos resultados e exibi-los de forma ideal. Além disso, também precisamos salvar em csv, para que nós consigamos ter uma análise gráfica de cada um.

Abaixo temos as funções que analisam e apresentam estatísticas resumidas dos produtos coletados — como preços mínimos, máximos, média, avaliação média e top 10 mais baratos — e permitem salvar os resultados em um arquivo CSV nomeado conforme a loja e o termo de busca, facilitando o armazenamento e a visualização dos dados processados.

In [8]:
def analyze_results(produtos, produto_buscado):
    """
    Recebe lista de dicts e exibe:
      • Total de produtos após filtro
      • Menor preço, maior preço, preço médio
      • Avaliação média (se houver)
      • Top 10 mais baratos (com título, preço, avaliação, link e relevância)
    Retorna o DataFrame ordenado (preço asc, similaridade desc).
    """
    if not produtos:
        print("❌ Nenhum produto encontrado!")
        return None

    df = pd.DataFrame(produtos)
    df_sorted = df.sort_values(["preco", "similaridade"], ascending=[True, False])

    print("\n" + "=" * 60)
    print(f"🎯 RESULTADOS PARA: '{produto_buscado.upper()}'")
    print("=" * 60)
    print(f"📦 Total de produtos (após filtro): {len(produtos)}")
    print(f"💰 Menor preço: R$ {df['preco'].min():.2f}")
    print(f"💸 Maior preço: R$ {df['preco'].max():.2f}")
    print(f"📊 Preço médio: R$ {df['preco'].mean():.2f}")

    if df["avaliacao"].notna().any():
        print(f"⭐ Avaliação média: {df['avaliacao'].mean():.1f}")

    print("\n🏆 TOP 10 MAIS BARATOS:")
    print("-" * 60)

    for i, (_, produto) in enumerate(df_sorted.head(10).iterrows(), start=1):
        print(f"\n{i}. 💰 R$ {produto['preco']:.2f}")
        print(f"   📦 {produto['titulo']}")
        if produto["avaliacao"] is not None and not (isinstance(produto["avaliacao"], float) and math.isnan(produto["avaliacao"])):
            stars = "⭐" * min(int(produto["avaliacao"]), 5)
            num_avals = produto["num_avaliacoes"] if (produto["num_avaliacoes"] is not None and not (isinstance(produto["num_avaliacoes"], float) and math.isnan(produto["num_avaliacoes"]))) else 0
            print(f"   {stars} {produto['avaliacao']:.1f} ({num_avals} avaliações)")
        print(f"   🔗 {produto['link']}")
        print(f"   📊 Relevância: {produto['similaridade']:.1%}")

    return df_sorted


def save_to_csv(df, produto_buscado, loja):
    """
    Salva o DataFrame em CSV com nome baseado na loja e no produto.
    """
    if df is not None and not df.empty:
        filename = f"{loja}_{produto_buscado.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False, encoding="utf-8")
        print(f"\n💾 Arquivo salvo: {filename}")

A função abaixo atua como um wrapper genérico que executa o scraping dinâmico da loja especificada, analisa os resultados, oferece a opção de salvá-los em CSV e retorna um DataFrame com os produtos relevantes — tudo de forma automatizada e personalizada conforme os parâmetros definidos pelo usuário. Ela consegue estabelecer uma conexão necessária para o scrapping funcionar.

In [9]:
def buscar_produtos_selenium(
    produto,
    loja="amazon",
    paginas=3,
    similaridade=0.3,
    extra_blacklist=None
):
    """
    Wrapper genérico que:
      1) Chama a função scrape_<loja>_selenium(produto, paginas, similaridade, extra_blacklist)
      2) Exibe resultados (analyze_results)
      3) Pergunta ao usuário se deseja salvar em CSV (save_to_csv)
      4) Retorna o DataFrame final
    """
    # Monta o nome da função de scraping dinamicamente
    nome_funcao = f"scrape_{loja}_selenium"
    funcao_scrape = globals().get(nome_funcao)

    if funcao_scrape is None:
        print(f"❌ Função de scraping não encontrada para a loja '{loja}'!")
        return None

    produtos = funcao_scrape(produto, paginas, similaridade, extra_blacklist)
    if produtos:
        df = analyze_results(produtos, produto)
        try:
            salvar = input("\n💾 Salvar em CSV? (s/n): ").strip().lower()
            if salvar in ["s", "sim", "y", "yes"]:
                save_to_csv(df, produto, loja)
        except:
            pass
        return df
    else:
        print("\n❌ Nenhum produto encontrado. Tente termos mais específicos ou ajuste a similaridade.")
        return None

# Explicação da função interativo_scraper()

Essa função implementa um fluxo interativo de scraping de preços com personalização por parte do usuário. Ela permite que o usuário escolha uma loja, busque um produto, refine os resultados com palavras indesejadas (blacklist) e salve os dados em CSV, seguindo os seguintes passos:

1. Entrada da loja – O usuário informa de qual loja deseja extrair os dados (ex.: "amazon").

2. Termo de busca – O usuário digita o nome do produto que deseja pesquisar.

3. Número de páginas – O usuário define quantas páginas do site a função deve varrer.

4. Similaridade mínima – O usuário informa o nível mínimo de similaridade entre os títulos encontrados e o termo buscado (valor entre 0.0 e 1.0).

5. Primeiro scraping – A função scrape_<loja>_selenium correspondente é executada, considerando apenas uma blacklist básica (ou nenhuma).

6. Análise inicial – Os dados são analisados e exibidos com a função analyze_results.

7. Refinamento com blacklist extra – O usuário pode digitar palavras para filtrar resultados indesejados (ex.: “capinha, usado”), e o scraping é refeito com essa nova blacklist. Isso pode ser repetido até o usuário decidir parar.

8. Exportação opcional – Após o refinamento, o usuário pode optar por salvar os resultados finais em um arquivo CSV.

9. Conclusão – O sistema finaliza com uma mensagem de agradecimento.






Aqui é onde tudo funciona junto. Essa função é a que realmente faz o usuário interagir com todas as outras. É o que realmente coloca todas as engrenagens para funcionar, perguntando tudo o que o usuario quer e colocando tudo em seu lugar, afinal, precisamos do usuário para fazer tudo isso funcionar E esse é o responsável por fazer a conexão.

In [10]:
def interativo_scraper():
    """
    Fluxo interativo:
      1) Pergunta loja (amazon, americanas, etc.)
      2) Pergunta termo de busca
      3) Pergunta quantas páginas varrer
      4) Pergunta similaridade mínima (0.0–1.0)
      5) Roda scrape_<loja>_selenium com apenas a BASE_BLACKLIST
      6) Exibe resultados iniciais (analyze_results)
      7) Pergunta se usuário quer adicionar palavras extras à blacklist
         • Se sim, solicita lista separada por vírgula
         • Reexecuta scrape_<loja>_selenium com blacklist base+extras
         • Exibe resultados refinados
      8) Repete passo 7 até usuário digitar “n”
      9) Por fim, pergunta se salva em CSV (save_to_csv)
      10) Exibe mensagem de conclusão
    """
    # 1) Loja
    loja = input("🛍️ Digite a loja para busca (ex.: amazon, americanas): ").strip().lower()
    if not loja:
        print("❌ Você precisa digitar uma loja válida. Tente novamente.")
        return

    # 2) Termo de busca
    produto = input(f"🔎 Digite o termo de busca para {loja.capitalize()}: ").strip()
    if not produto:
        print("❌ Você precisa digitar um termo válido. Tente novamente.")
        return

    # 3) Quantas páginas?
    paginas = None
    while paginas is None:
        try:
            paginas = int(input("📄 Quantas páginas deseja varrer? (ex.: 3): ").strip())
            if paginas <= 0:
                print("❌ Informe um número inteiro positivo.")
                paginas = None
        except:
            print("❌ Valor inválido. Por favor, digite um número inteiro.")
            paginas = None

    # 4) Similaridade mínima
    min_similarity = None
    while min_similarity is None:
        try:
            ms = float(input("🔢 Qual similaridade mínima (0.0 a 1.0)? (ex.: 0.3): ").strip().replace(",", "."))
            if 0.0 <= ms <= 1.0:
                min_similarity = ms
            else:
                print("❌ Informe um valor entre 0.0 e 1.0.")
        except:
            print("❌ Valor inválido. Digite algo como 0.3 ou 0.5.")
            min_similarity = None

    # 5) Primeira chamada (blacklist base vazia ou mínima)
    nome_funcao = f"scrape_{loja}_selenium"
    funcao_scrape = globals().get(nome_funcao)
    if funcao_scrape is None:
        print(f"❌ Função de scraping não encontrada para a loja '{loja}'!")
        return

    print("\n---- RESULTADOS INICIAIS (apenas com blacklist base) ----")
    resultados = funcao_scrape(produto, paginas, min_similarity, extra_blacklist=[])
    df = analyze_results(resultados, produto)

    # 6) Loop para adicionar palavras extras à blacklist
    combined_extra = []
    while True:
        resp = input("\n🛑 Deseja adicionar palavras extras para filtrar resultados? (s/n): ").strip().lower()
        if resp in ["n", "nao", "não", "no"]:
            break
        elif resp in ["s", "sim", "yes", "y"]:
            extras_str = input("✏️ Digite as palavras (separadas por vírgula) que deseja bloquear: ").strip()
            if not extras_str:
                print("❌ Você não digitou nada. Tente novamente.")
                continue

            novas_palavras = [w.strip().lower() for w in extras_str.split(",") if w.strip()]
            combined_extra.extend(novas_palavras)
            combined_extra = list(set(combined_extra))  # Remover duplicatas

            print(f"\n---- RESULTADOS REFINADOS (blacklist extra: {combined_extra}) ----")
            resultados = funcao_scrape(produto, paginas, min_similarity, extra_blacklist=combined_extra)
            print("\n📦 Produtos brutos retornados:", len(resultados))
            for p in resultados:
                print(p)
            df = analyze_results(resultados, produto)
        else:
            print("❌ Resposta inválida. Digite 's' para sim ou 'n' para não.")

    # 7) Perguntar se salva em CSV
    try:
        salvar = input("\n💾 Deseja salvar o resultado final em CSV? (s/n): ").strip().lower()
        if salvar in ["s", "sim", "yes", "y"]:
            save_to_csv(df, produto, loja)
    except:
        pass

    print("\n✅ Processo concluído. Obrigado por usar o Garimpa AI!")


# **Parte prática**
Nesta etapa, o usuário pode utilizar efetivamente o projeto, personalizando sua busca de acordo com suas preferências e critérios específicos

Rode a célula abaixo para usar nosso projeto:

In [ ]:
interativo_scraper()
#Se der um erro de perda de conexao ou algo assim, fechar o collab, e abrir denovo, rodando novamente todas as células.


---- RESULTADOS INICIAIS (apenas com blacklist base) ----
🔍 Buscando (Selenium) por: 'Monitor 1440p'  |  Páginas: 6  |  Sim. mínima: 30.0%
------------------------------------------------------------
· Página 1... URL de busca: https://www.amazon.com.br/s?k=Monitor+1440p&page=1
48 encontrados.
· Página 2... URL de busca: https://www.amazon.com.br/s?k=Monitor+1440p&page=2
48 encontrados.
· Página 3... URL de busca: https://www.amazon.com.br/s?k=Monitor+1440p&page=3
48 encontrados.
· Página 4... URL de busca: https://www.amazon.com.br/s?k=Monitor+1440p&page=4
48 encontrados.
· Página 5... URL de busca: https://www.amazon.com.br/s?k=Monitor+1440p&page=5
60 encontrados.
· Página 6... URL de busca: https://www.amazon.com.br/s?k=Monitor+1440p&page=6
48 encontrados.

🎯 RESULTADOS PARA: 'MONITOR 1440P'
📦 Total de produtos (após filtro): 242
💰 Menor preço: R$ 132.00
💸 Maior preço: R$ 13565.00
📊 Preço médio: R$ 2492.11
⭐ Avaliação média: 4.4

🏆 TOP 10 MAIS BARATOS:
-----------------------------

# Análise de dados e resultados

Esta parte do código analisa os preços dos produtos em diferentes lojas a partir de arquivos em CSV.


Especificamente esse script define uma função que recebe listas de nomes de lojas e caminhos de arquivos CSV,
processa esses arquivos para extrair dados dos produtos e preços e gerar gráficos
comparativos bem como identificar a loja com os melhores preços médios em geral.

In [ ]:
# importação das bibliotecas
import matplotlib.pyplot as plt
import seaborn as sns

Gerando a função

In [ ]:
def analise_precos_lojas(nomes_lojas, csv_files):
    """
    Analisa preços de produtos de várias lojas a partir de arquivos CSV.

    Argumentos:
        nomes_lojas (list): Uma lista de strings contendo os nomes das lojas.
        csv_files (list): Uma lista de strings contendo os caminhos para os arquivos CSV,
                          na mesma ordem dos nomes das lojas.

    Returns:
        None. A função exibe os gráficos e imprime as análises diretamente.

    Raises:
        ValueError: Se o número de nomes de lojas não corresponder ao número de arquivos CSV.
        FileNotFoundError: Se algum dos arquivos CSV não for encontrado.
        KeyError: Se as colunas esperadas ('titulo', 'Preço') não forem encontradas nos CSVs.
    """

    if len(nomes_lojas) != len(csv_files):
        raise ValueError("O número de nomes de lojas deve ser igual ao número de arquivos CSV.")

    all_data = []

    print("Iniciando o processamento dos arquivos CSV...")
    for nome_loja, file_path in zip(nomes_lojas, csv_files):
        print(f"Processando arquivo da loja: {nome_loja} ({os.path.basename(file_path)})...", end=" ")
        try:
            # Tenta ler o CSV com diferentes separadores comuns
            try:
                df_store = pd.read_csv(file_path, sep=',')
            except pd.errors.ParserError:
                try:
                    df_store = pd.read_csv(file_path, sep=';')
                except Exception as e:
                    print(f"\nErro ao ler {file_path} com separadores ',' e ';': {e}")
                    continue # Pula para o próximo arquivo

            # Verifica se as colunas essenciais existem
            if 'titulo' not in df_store.columns or 'preco' not in df_store.columns:
                 # Tenta encontrar colunas com nomes ligeiramente diferentes (case-insensitive)
                nome_col = next((col for col in df_store.columns if col.lower() == 'titulo'), None)
                preco_col = next((col for col in df_store.columns if col.lower() == 'preco'), None)

                if not nome_col or not preco_col:
                    print(f"\nErro: Colunas 'titulo' e/ou 'preco' não encontradas em {os.path.basename(file_path)}. Colunas disponíveis: {list(df_store.columns)}")
                    continue # Pula para o próximo arquivo
                else:
                    # Renomeia as colunas encontradas para o padrão esperado
                    df_store.rename(columns={nome_col: 'titulo', preco_col: 'preco'}, inplace=True)

            # Seleciona e copia as colunas necessárias para evitar SettingWithCopyWarning
            df_processed = df_store[['titulo', 'preco']].copy()
            df_processed['Loja'] = nome_loja

            # Limpeza da coluna de preco
            # Remove 'R$', espaços em branco, substitui ',' por '.' e converte para float
            if df_processed['preco'].dtype == 'object': # Processa apenas se for string
                df_processed['preco'] = df_processed['preco'].astype(str) # Garante que é string
                df_processed['preco'] = df_processed['preco'].str.replace(r'[R$\s]', '', regex=True)
                df_processed['preco'] = df_processed['preco'].str.replace(',', '.', regex=False)
                # Tenta converter para numérico, tratando erros
                df_processed['preco'] = pd.to_numeric(df_processed['preco'], errors='coerce')

            # Remove linhas onde o preço não pôde ser convertido (NaN)
            df_processed.dropna(subset=['preco'], inplace=True)

            all_data.append(df_processed)
            print("OK")

        except FileNotFoundError:
            print(f"\nErro: Arquivo não encontrado - {file_path}")
            # Decide se quer parar ou continuar
            # raise FileNotFoundError(f"Arquivo não encontrado: {file_path}")
            continue # Continua para o próximo arquivo
        except Exception as e:
            print(f"\nOcorreu um erro inesperado ao processar {file_path}: {e}")
            continue # Continua para o próximo arquivo

    if not all_data:
        print("\nNenhum dado foi processado com sucesso. Verifique os arquivos CSV e seus formatos.")
        return

    # Combina todos os dataframes em um só
    df_combined = pd.concat(all_data, ignore_index=True)
    print("\nDados combinados com sucesso.")
    print(f"Total de registros processados: {len(df_combined)}")
    print(f"Lojas incluídas na análise: {df_combined['Loja'].unique().tolist()}")

    # --- Geração dos Gráficos Top 10 por Loja ---
    print("\nGerando gráficos Top 10 produtos mais baratos por loja...")
    plt.style.use('seaborn-v0_8-whitegrid')

    for nome_loja in df_combined['Loja'].unique():
        df_store_filtered = df_combined[df_combined['Loja'] == nome_loja].copy()
        # Ordena por preço e pega os 10 mais baratos
        top_10_cheapest = df_store_filtered.sort_values(by='preco', ascending=True).head(10)

        if top_10_cheapest.empty:
            print(f"Não há dados suficientes para gerar o gráfico da loja {nome_loja}.")
            continue

        plt.figure(figsize=(12, 7)) # Ajusta o tamanho da figura
        barplot = sns.barplot(x='titulo', y='preco', data=top_10_cheapest, palette='viridis')

        plt.title(f'Top 10 Produtos Mais Baratos - {nome_loja}', fontsize=16)
        plt.xlabel('Nome do Produto', fontsize=12)
        plt.ylabel('preco (R$)', fontsize=12)
        plt.xticks(rotation=45, ha='right', fontsize=10) # Rotaciona os nomes dos produtos
        plt.yticks(fontsize=10)

        # Adiciona os valores exatos nas barras
        for container in barplot.containers:
            barplot.bar_label(container, fmt='R$ %.2f', fontsize=9, padding=3)

        plt.tight_layout() # Ajusta o layout para evitar sobreposição
        plt.show()
        print(f"Gráfico da loja {nome_loja} gerado.")

    # --- Análise de Loja Mais Barata em Média (Geral) ---
    print("\nCalculando a loja mais barata em média...")

    # Calcula o preço médio geral por loja
    media_por_loja = df_combined.groupby('Loja')['preco'].mean().reset_index()

    # Encontra a loja com menor média
    loja_mais_barata = media_por_loja.loc[media_por_loja['preco'].idxmin()]
    print("\n--- Loja Mais Barata em Média ---")
    print(f"A loja mais barata em média para os produtos analisados é: {loja_mais_barata['Loja']} (Preço médio: R$ {loja_mais_barata['preco']:.2f})")
    print("--------------------------------------------------")

    print("\nAnálise concluída.")

Solicitando dados do usuário (testando a função de análise de dados)

In [ ]:
Lista_lojas_str = input("🛍️ Digite a lista de lojas (separadas por vírgula): ")
Lista_lojas = [loja.strip() for loja in Lista_lojas_str.split(',') if loja.strip()]

caminho_lojas_str = input("📄 Digite a lista de caminhos para os arquivos CSV (separados por vírgula): ")
caminho_lojas = [caminho.strip() for caminho in caminho_lojas_str.split(',') if caminho.strip()]

analise_precos_lojas(Lista_lojas, caminho_lojas)


# Conclusão


Em síntese, num ambiente de preços voláteis e excesso de ofertas online, contar apenas com pesquisa manual já não atende às exigências de agilidade e precisão do consumidor moderno. A automação da coleta de dados — via scraper — torna-se, portanto, uma aliada estratégica: consolida em poucos segundos informações dispersas em marketplaces como Amazon e Americanas, revela variações e promoções ocultas à navegação comum e amplia o poder de barganha de quem compra.

Ao transformar o tradicional “garimpo” em um processo estruturado, transparente e escalável, essa abordagem reduz custos de tempo e dinheiro, melhora a experiência de compra e incentiva a concorrência saudável entre vendedores. Assim, investir em uma ferramenta de extração automática não é apenas uma solução técnica; é um passo essencial para democratizar o acesso ao melhor preço e fortalecer a tomada de decisões no comércio brasileiro.

As principais dificuldades enfrentadas no desenvolvimento do projeto envolveram a criação de um ambiente interativo que permitisse ao usuário personalizar sua busca conforme suas preferências — como, por exemplo, a definição de uma 'lista negra' personalizada com base nos resultados extraídos via web scraping. Além disso, identificamos desafios na previsão e tratamento de erros decorrentes do uso incorreto da aplicação, como a digitação de palavras com erros ortográficos, variações de acentuação e uso de letras maiúsculas/minúsculas.

Para contornar parte dessas dificuldades, adotamos o uso de expressões regulares (RegEx), que nos permitiram tratar os textos de forma mais flexível, padronizando os dados e facilitando a filtragem e comparação dos resultados obtidos.

Apesar das limitações de tempo e da simplicidade inicial da solução, estamos satisfeitos com os resultados alcançados, que já se mostram eficazes na resolução do problema proposto. Isso reforça nossa motivação em dar continuidade ao projeto, aprimorando suas funcionalidades, ampliando o número de lojas analisadas e abordando com mais profundidade os aspectos que ainda precisam de melhorias.

Para o futuro, planejamos desenvolver uma interface mais intuitiva e acessível, com o objetivo de democratizar o uso da ferramenta e torná-la acessível a usuários sem conhecimento técnico em programação. Possívelmente implementando uma extrutura mais robusta de Front-end e talvez implementando outros Marketplaces com um scrapping de maior eficiência.

